In [ ]:
import os
import sys

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

    colab_src_dir = "/content/drive/MyDrive/Luca/research/unsupervised-feature-research/src"
    os.chdir(colab_src_dir)
    if colab_src_dir not in sys.path:
        sys.path.insert(0, colab_src_dir)

### Setup

In [ ]:
import sys
sys.path.insert(0, '.')

import copy
import math
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from transformers import AutoFeatureExtractor, AutoModel, AutoModelForImageClassification

from models import Autoencoder, K_Sparse_AE, WTA_FC_AE, WTA_CONV_AE
from datasets import get_data_loader, get_flattened_size, get_patch_shape

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device}")

In [ ]:
wandb.init(
    project="filter-deduplicate",
    config={
        "k_population": 0.05,
        "k_spatial": 0.2,
        "k_lifetime": 0.2,
        "epochs": 30,
        "batch_size": 128,
        "bottleneck_dim": 250,
        "lr": 0.001,
        "dataset": "cifar10_color",
        "autoencoder_type": "normal",
        "num_angle_samples": 500,
        "angle_pool_size": 2000,
        "max_filters_viz": 300,
    },
)

### Dataset

In [ ]:
os.makedirs('../data', exist_ok=True)

In [ ]:
cfg = wandb.config

data_loader = get_data_loader(cfg.dataset, train=True, batch_size=cfg.batch_size)
test_loader = get_data_loader(cfg.dataset, train=False, batch_size=cfg.batch_size)
input_dim   = get_flattened_size(cfg.dataset)

### Autoencoders

In [ ]:
def get_autoencoder(autoencoder_type: str):
    cfg = wandb.config
    match autoencoder_type:
        case "normal":
            return Autoencoder(
                dim=(input_dim, cfg.bottleneck_dim),
            ).to(device)

        case "sparse":
            return K_Sparse_AE(
                dim=(input_dim, cfg.bottleneck_dim),
                k_population=cfg.k_population,
                total_epochs=cfg.epochs,
                dataset_size=len(data_loader.dataset),
            ).to(device)

        case "wta_fc":
            return WTA_FC_AE(
                dim=(input_dim, cfg.bottleneck_dim),
                k_lifetime=cfg.k_lifetime,
            ).to(device)

        case "wta_conv":
            try:
                c, h, w = get_patch_shape(cfg.dataset)
            except ValueError:
                s = int(math.sqrt(input_dim))
                c, h, w = 1, s, s

            return WTA_CONV_AE(
                dim=(c, h, w, cfg.bottleneck_dim),
                k_spatial=cfg.k_spatial,
                k_population=cfg.k_lifetime,
                total_epochs=cfg.epochs,
                dataset_size=len(data_loader.dataset),
            ).to(device)

        case _:
            raise ValueError(f"Unknown autoencoder_type={autoencoder_type!r}.")

In [ ]:
cfg = wandb.config
autoencoder = get_autoencoder(cfg.autoencoder_type)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=cfg.lr)

### Training

In [ ]:
def compute_test_loss(cur_epoch: int) -> float:
    autoencoder.eval()
    test_loss_sum = 0.0
    test_n = 0
    with torch.no_grad():
        for batch_inputs, _ in test_loader:
            batch_inputs = batch_inputs.to(device)
            if autoencoder.uses_k_population:
                x_recon = autoencoder(
                    batch_inputs,
                    epoch=cur_epoch,
                    inputs_processed_in_epoch=0,
                )
            else:
                x_recon = autoencoder(batch_inputs)
            test_loss_sum += criterion(x_recon, batch_inputs).item() * batch_inputs.shape[0]
            test_n += batch_inputs.shape[0]
    autoencoder.train()
    return test_loss_sum / max(test_n, 1)

In [ ]:
losses = []
model_checkpoints = []

autoencoder.train()

cfg = wandb.config
ae_key    = cfg.autoencoder_type
epochbar  = tqdm(range(cfg.epochs), desc="Epochs")
n_batches = len(data_loader)
log_every = max(1, n_batches // 4)

for epoch in epochbar:
    epoch_loss                = 0.0
    inputs_processed_in_epoch = 0

    for step, (batch_inputs, _) in enumerate(data_loader):
        batch_size_cur = batch_inputs.shape[0]
        batch_inputs   = batch_inputs.to(device)
        optimizer.zero_grad()

        if autoencoder.uses_k_population:
            x_recon = autoencoder(
                batch_inputs,
                epoch=epoch,
                inputs_processed_in_epoch=inputs_processed_in_epoch,
            )
        else:
            x_recon = autoencoder(batch_inputs)

        inputs_processed_in_epoch += batch_size_cur

        loss = criterion(x_recon, batch_inputs)
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        epoch_loss += batch_loss

        if (step + 1) % log_every == 0:
            avg = epoch_loss / (step + 1)
            losses.append(avg)
            wandb.log({
                "train/loss_running": avg,
                "test/loss": compute_test_loss(epoch),
                "epoch": epoch,
            })

        epochbar.set_postfix({ae_key: f"{batch_loss:.4f}"})

    epoch_mean = epoch_loss / n_batches
    test_loss_mean = compute_test_loss(epoch)

    wandb.log({
        "train/loss": epoch_mean,
        "test/loss": test_loss_mean,
        "epoch": epoch,
    })

    model_checkpoints.append({ae_key: copy.deepcopy(autoencoder)})

### Visualizations

In [ ]:
def log_figure_wandb(name: str, fig=None, close: bool = True) -> None:
    if fig is None:
        fig = plt.gcf()
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    rgb = rgba[..., :3]
    wandb.log({name: wandb.Image(rgb)})
    if close:
        plt.close(fig)

#### 1. Dataset

In [ ]:
cfg = wandb.config
patches_batch, _ = next(iter(data_loader))
n_show = min(25, patches_batch.size(0))

try:
    C, H, W = get_patch_shape(cfg.dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s

patches_vis = patches_batch[:n_show].reshape(-1, C, H, W)
grid_patches = make_grid(patches_vis, nrow=5, normalize=True, scale_each=True, padding=2)
fig = plt.figure(figsize=(8, 8))

plt.imshow(grid_patches.permute(1, 2, 0).clamp(0, 1))
plt.axis("off")
plt.title(f"Sample {cfg.dataset} patches (n={n_show})")
plt.tight_layout()
log_figure_wandb("visualizations/dataset/sample_patches", fig, close=False)
plt.show()

#### 2. Encoder weights

In [ ]:
visualize_epoch = cfg.epochs - 1

try:
    C, H, W = get_patch_shape(cfg.dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s

cp = model_checkpoints[visualize_epoch]
m = cp[cfg.autoencoder_type]
m.eval()

x_probe, _ = next(iter(data_loader))
x_probe = x_probe[:1].to(device)

with torch.no_grad():
    if m.uses_k_population:
        m(x_probe, epoch=visualize_epoch, inputs_processed_in_epoch=0)
    else:
        m(x_probe)

def mask_linear_encoder_rows(W: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    m_row = mask.view(-1)
    return W * m_row.unsqueeze(1)

def mask_conv_encoder_out(W: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    m_row = mask.view(-1)
    return W * m_row.view(-1, 1, 1, 1)

mask_t = m.last_filter_mask[0].cpu()
max_filters = min(cfg.max_filters_viz, m.detached_encoder_weights.shape[0])

if m.is_convolutional:
    W_enc = mask_conv_encoder_out(m.detached_encoder_weights.cpu(), mask_t)
    kernels = W_enc[:max_filters]
else:
    W_enc = mask_linear_encoder_rows(m.detached_encoder_weights.cpu(), mask_t)
    kernels = W_enc[:max_filters].reshape(-1, C, H, W)

grid_enc = make_grid(kernels, nrow=10, normalize=True, scale_each=True, padding=1)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid_enc.permute(1, 2, 0).clamp(0, 1))
ax.axis("off")
ax.set_title(f"{cfg.autoencoder_type} - masked encoder weights")
fig.suptitle(f"Epoch {visualize_epoch}, probe batch n=1")
plt.tight_layout()
log_figure_wandb(f"visualizations/encoder_weights/{cfg.autoencoder_type}", fig, close=False)
plt.show()

#### 3. Decoder weights

In [ ]:
visualize_epoch = cfg.epochs - 1

try:
    C, H, W = get_patch_shape(cfg.dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s

cp = model_checkpoints[visualize_epoch]
m = cp[cfg.autoencoder_type]
max_filters = cfg.max_filters_viz

W_dec = m.detached_decoder_weights.cpu()
if m.is_convolutional:
    kernels_dec = W_dec[:max_filters]
else:
    kernels_dec = W_dec.T[:max_filters].reshape(-1, C, H, W)

grid_dec = make_grid(kernels_dec, nrow=10, normalize=True, scale_each=True, padding=1)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid_dec.permute(1, 2, 0).clamp(0, 1))
ax.axis("off")
ax.set_title(f"{cfg.autoencoder_type} - decoder weights")
plt.suptitle(f"Epoch {visualize_epoch}")
plt.tight_layout()
log_figure_wandb(f"visualizations/decoder_weights/{cfg.autoencoder_type}", fig, close=False)
plt.show()

#### 4. Reconstructions (25 random patches)

In [ ]:
visualize_epoch = cfg.epochs - 1

cp = model_checkpoints[visualize_epoch]
m  = cp[cfg.autoencoder_type]
m.eval()

batch_inputs, _ = next(iter(data_loader))
idx = torch.randperm(batch_inputs.shape[0])[:25]
inputs_25 = batch_inputs[idx].to(device)

with torch.no_grad():
    if m.uses_k_population:
        recon = m(
            inputs_25,
            epoch=visualize_epoch,
            inputs_processed_in_epoch=0,
        ).cpu()
    else:
        recon = m(inputs_25).cpu()

try:
    C, H, W = get_patch_shape(cfg.dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s

orig_vis = inputs_25.cpu().reshape(-1, C, H, W)
rec_vis = recon.reshape(-1, C, H, W)

grid_orig = make_grid(orig_vis, nrow=5, normalize=True, scale_each=True, padding=2)
grid_rec = make_grid(rec_vis, nrow=5, normalize=True, scale_each=True, padding=2)

fig, axes = plt.subplots(2, 1, figsize=(8, 8))

axes[0].imshow(grid_orig.permute(1, 2, 0).clamp(0, 1))
axes[0].axis("off")
axes[0].set_title("Original patches (25)")

axes[1].imshow(grid_rec.permute(1, 2, 0).clamp(0, 1))
axes[1].axis("off")
axes[1].set_title(f"{cfg.autoencoder_type} reconstructions")

plt.tight_layout()
log_figure_wandb(f"visualizations/reconstructions/{cfg.autoencoder_type}", fig, close=False)
plt.show()